In [ ]:
# Simulation code for example #1. 
# This can be easily modified to work for example #2

import numpy as np
import matplotlib.pyplot as plt

# ===============================================================
# Given information:
# ===============================================================

a = 1
b = 1
c = 1
d = 0
R = 0.1      # Measurement noise variance
Q = 0.0001   # Process noise variance
S = 0

# Length of signals
L = 200

# Set random seed for reproducibility
np.random.seed(65)

# ===============================================================
# Simulating process: Here, the state is treated as an unknown and 
# used only to simulate the process
# ===============================================================

x = np.zeros(L + 1)      # True (hidden) state
y = np.zeros(L)          # Measured signal
u = np.zeros(L)          # Control input
w = np.zeros(L)          # Process noise
v = np.zeros(L)          # Measurement noise

x[0] = 20                # Initial true state

for k in range(L):       # Iterating over L time-steps

    # Process and measurement models implemented as per Results 1 and 2:

    u[k] = 0

    w[k] = np.random.randn() * np.sqrt(Q)   # Gaussian distributed process noise
    v[k] = np.random.randn() * np.sqrt(R)   # Gaussian distributed measurement noise

    x[k + 1] = a * x[k] + b * u[k] + w[k]
    y[k]     = c * x[k] + d * u[k] + v[k]

# ===============================================================
# Kalman filter Implementation:
# ===============================================================

# Initial condition for P(k|k-1) as defined in result ___
P = np.zeros(L + 1)

xf = np.zeros(L + 1)    # Filtered estimate
xe = np.zeros(L + 1)    # One-step ahead estimate 

xf[0] = y[0]            # Initial guess for filtered estimate - equal to initial measurement
xe[0] = y[0]            # Initial guess for one-step ahead estimate - equal to initial measurement

Pk = np.zeros(L)        # To store P(k|k-1) for each step

for k in range(L):       # Iterating over L time-steps

    # ---------------------------------------------------------------
    # Filtered estimate: as per Figure 3
    # ---------------------------------------------------------------

    # Kalman gain:
    K1 = c * P[k] / (R + c**2 * P[k])

    # Minimum variance:
    Pk[k] = (1 - K1 * c) * P[k]

    # Estimator equation:
    xf[k] = xe[k] + K1 * (y[k] - xe[k] * c - d * u[k])

    # ---------------------------------------------------------------
    # One-step ahead estimate: as per Figure 4
    # ---------------------------------------------------------------

    # Kalman gain:
    K = (S + a * c * P[k]) / (R + c**2 * P[k])

    # Estimator equation:
    xe[k + 1] = a * xe[k] + b * u[k] + K * (y[k] - xe[k] * c - d * u[k])

    # Minimum variance:
    P[k + 1] = a**2 * P[k] + Q - K * (S + a * c * P[k])

# ===============================================================
# Plotting results:
# ===============================================================

plt.figure(1)
plt.plot(range(1, len(y) + 1), y, color=(0, 0.5, 0), label='Measured signal')
plt.legend()
plt.xlabel('Time step - k')
plt.ylabel('Temperature [°C]')
plt.grid(True)
plt.xlim([1, L])
plt.ylim([18.5, 21.5])

plt.figure(2)
plt.plot(range(1, len(y) + 1), y, color=(0, 0.5, 0), label='Measured signal [y]')
plt.plot(range(1, len(xf) + 1), xf, color=(0, 0, 1), linewidth=1.5,
         label='Filtered Estimate [xf]')
plt.plot(range(1, len(xe) + 1), xe, color=(0.5, 0, 0), linewidth=1.5,
         label='One step ahead Estimate [xe]')
plt.legend()
plt.xlabel('Time step - k')
plt.ylabel('Temperature [°C]')
plt.grid(True)
plt.xlim([1, L])
plt.ylim([18.5, 21.5])

plt.figure(3)
plt.plot(range(1, len(x) + 1), x, 'k', label='Actual unknown state [x]')
plt.plot(range(1, len(xf) + 1), xf, color=(0, 0, 1), linewidth=1.5,
         label='Filtered Estimate [xf]')
plt.plot(range(1, len(xe) + 1), xe, color=(0.5, 0, 0), linewidth=1.5,
         label='One step ahead Estimate [xe]')
plt.legend()
plt.xlabel('Time step - k')
plt.ylabel('Temperature [°C]')
plt.grid(True)
plt.xlim([1, L])
plt.ylim([18.5, 21.5])

plt.show()

Ejemplo tomado de Kalman Filters Demystified — The Algorithm Behind Moon Landings por Maxwell's Demon (https://pub.towardsai.net/kalman-filters-demystified-the-algorithm-behind-moon-landings-6fcf46433a50)

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from pykalman import KalmanFilter

data = yf.download(
    "AAPL",
    start="2020-01-01",
    end="2021-01-01",
    auto_adjust=False,
    progress=False,
    threads=False,
)
prices = data["Close"]

kf = KalmanFilter(
    transition_matrices=[1],
    observation_matrices=[1],
    initial_state_mean=prices.iloc[0],
    initial_state_covariance=1,
    observation_covariance=1,
    transition_covariance=0.01
)

state_means, _ = kf.filter(prices.values)
kalman_series = pd.Series(state_means.flatten(), index=prices.index)

ma30 = prices.rolling(30).mean()

plt.figure(figsize=(12, 6))
plt.plot(prices, label="Price", alpha=0.5)
plt.plot(kalman_series, label="Kalman Filter")
plt.plot(ma30, label="30-Day Moving Average")
plt.legend()
plt.show()

ejemplo tomado de Emma Kirsten, Why the Kalman Filter Beats Moving Averages in Trading https://medium.com/coding-nexus/why-the-kalman-filter-beats-moving-averages-in-trading-36d215a3f1b7
